# PlantSeg Round-2 -- Notebook J: null-model baselines that were never actually computed (Table 12 + R4 Concern 2)

**Why this notebook exists.** An audit of every file on this project found that two
analyses currently stated as completed, with specific numbers and p-values, have no
computational backing anywhere: the centre-prior (Gaussian-blob) baseline column in
Table 12 (all 7 rows, including the caption's own summary numbers), and the
model-parameter-randomisation sanity check described in the response letter (Reviewer 4,
Concern 2: "trained IoU 0.267 vs. randomised-weights IoU 0.156, p = 1.2e-22"). Neither
exists as a notebook, script, or CSV anywhere in the project. This notebook computes both
for real.

**What this notebook does, concretely.**
1. **Centre-prior baseline.** A fixed, image-size-matched 2D Gaussian centred on the
   frame (sigma = 0.2 x min(H, W), a standard "center-bias" scale from the saliency
   literature -- this is a stated design choice, not fit to the data) stands in for the
   attribution map, run through the exact same `cam_gt_metrics` thresholding code used for
   every real attribution in this paper. Computed for the classifier and all 6 segmenters
   on the same 300-image stratified subset as Table 10/11/12.
2. **SegFormer-B2, for real this time.** Every earlier attempt at this check (including
   NotebookA's own chance-baseline run) failed to load the SegFormer-B2 checkpoint because
   `transformers` was unpinned. This notebook pins `transformers==4.46.3` -- the version
   NotebookC already proved loads this exact checkpoint with 0 missing/0 unexpected keys
   -- so SegFormer-B2 gets a real row in both the random-attribution and centre-prior
   columns for the first time.
3. **Model-parameter-randomisation sanity check.** A second, freshly and fully randomly
   initialised ConvNeXtV2-Tiny (same architecture, no trained weights) has its Grad-CAM
   computed on the same 300 images as the trained classifier, and the two are compared
   with a paired Wilcoxon test -- the standard sanity check (Adebayo et al., 2018) for
   whether an attribution method is actually sensitive to what the model learned.
4. **Random-attribution baseline, recomputed fresh too** (not just reused from
   NotebookA's CSV), so this notebook is self-contained and the real-IoU/real-hit-rate
   numbers it reports for all 7 rows are independently re-verified, not copied.

**A caveat worth stating plainly:** the centre-prior baseline numbers this notebook
produces are a genuine computation, but the sigma choice above is a design decision.
They will almost certainly not exactly match the 0.284/0.467 currently sitting in
`paper.tex` -- those numbers were never actually computed, so there is nothing for this
notebook to "reproduce." Whatever this notebook reports is the real number, and
`paper.tex` and `Author_Response_2.docx` will need updating to match it, not the other
way around.

## Setup -- pins transformers==4.46.3 so SegFormer-B2 finally loads

In [1]:
import importlib, os, subprocess, sys, time

def heartbeat(msg):
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

heartbeat("Notebook J starting: null-model baselines (centre-prior + randomisation)")

for pkg, import_name in [("segmentation-models-pytorch", "segmentation_models_pytorch"),
                          ("grad-cam", "pytorch_grad_cam"),
                          ("timm", "timm")]:
    try:
        importlib.import_module(import_name)
        heartbeat(f"{import_name} already importable, skipping pip install")
    except ImportError:
        heartbeat(f"installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        heartbeat(f"{pkg} installed")

# huggingface_hub must be pinned to a version contemporary with transformers==4.46.3
# (pre-Xet) too -- otherwise Kaggle's current huggingface_hub/hf_xet combination breaks
# from_pretrained() with "ImportError: cannot import name 'XetDownloadProgressReporter'
# from 'huggingface_hub.utils._xet_progress_reporting'", a version-skew bug in
# huggingface_hub's newer Xet download backend unrelated to the checkpoint itself (this
# is exactly what Notebook I's first Kaggle run hit). Disable Xet outright, purge any
# already-imported transformers/huggingface_hub/hf_xet modules, and install both
# packages together so the resolved versions are self-consistent.
os.environ["HF_HUB_DISABLE_XET"] = "1"
TRANSFORMERS_PIN = "4.46.3"
HF_HUB_PIN = "0.26.2"
for mod in list(sys.modules):
    if mod.split(".")[0] in ("transformers", "huggingface_hub", "hf_xet"):
        del sys.modules[mod]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet"])
heartbeat(f"installing huggingface_hub=={HF_HUB_PIN} transformers=={TRANSFORMERS_PIN} ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                f"huggingface_hub=={HF_HUB_PIN}", f"transformers=={TRANSFORMERS_PIN}"],
               check=True)
heartbeat(f"huggingface_hub=={HF_HUB_PIN} / transformers=={TRANSFORMERS_PIN} installed")

heartbeat("imports starting")
import json, glob, random, warnings, copy
from pathlib import Path
from collections import defaultdict
import numpy as np, pandas as pd
from PIL import Image, ImageDraw
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import timm
import segmentation_models_pytorch as smp
import huggingface_hub as _hfh
from transformers import SegformerForSemanticSegmentation
import transformers as _tf
from pytorch_grad_cam import GradCAM
from scipy.stats import wilcoxon
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
heartbeat(f"imports done, device={device}, transformers=={_tf.__version__}, "
          f"huggingface_hub=={_hfh.__version__}")
assert _tf.__version__ == TRANSFORMERS_PIN, (
    f"pinned transformers=={TRANSFORMERS_PIN} but {_tf.__version__} is what actually "
    f"imported -- stop and check the Kaggle environment before trusting the "
    f"SegFormer-B2 result below.")
assert _hfh.__version__ == HF_HUB_PIN, (
    f"pinned huggingface_hub=={HF_HUB_PIN} but {_hfh.__version__} is what actually "
    f"imported -- stop and check the Kaggle environment before trusting the "
    f"SegFormer-B2 result below.")

[15:50:10] Notebook J starting: null-model baselines (centre-prior + randomisation)
[15:50:10] installing segmentation-models-pytorch ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.7 MB/s eta 0:00:00
[15:50:15] segmentation-models-pytorch installed
[15:50:15] installing grad-cam ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.5/44.5 kB 2.1 MB/s eta 0:00:00
[15:50:24] grad-cam installed
[15:50:41] timm already importable, skipping pip install
[15:50:42] installing huggingface_hub==0.26.2 transformers==4.46.3 ...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 77.4 MB/s eta 0:00:00
[15:50:58] huggingface_hub==0.26.2 / transformers==4.46.3 installed
[15:50:58] imports starting


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
litellm 1.85.7 requires tokenizers<1.0,>=0.21.0, but you have tokenizers 0.20.3 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.26.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.26.2 which is incompatible.


[15:51:20] imports done, device=cuda, transformers==4.46.3, huggingface_hub==0.26.2


## Config -- dataset, checkpoints, classifier/segmenter builders (same code as NotebookA)

In [2]:
def find_data_root():
    for cand in [Path("/kaggle/input/datasets/alenmaruf/plantseg/plantseg"),
                 Path("/kaggle/input/plantseg"), Path("plantseg"), Path(".")]:
        if (cand/"Metadata.csv").exists(): return cand
    hits = glob.glob("/kaggle/input/**/Metadata.csv", recursive=True)
    assert hits, "Metadata.csv not found anywhere under /kaggle/input -- attach the PlantSeg dataset."
    return Path(hits[0]).parent

DATA_ROOT = find_data_root()
heartbeat(f"DATA_ROOT = {DATA_ROOT}")

def find_ckpt_dir(required_names):
    for d in glob.glob("/kaggle/input/*"):
        if all((Path(d)/n).exists() for n in required_names): return Path(d)
    for n in required_names:
        hits = glob.glob(f"/kaggle/input/**/{n}", recursive=True)
        if hits: return Path(hits[0]).parent
    raise FileNotFoundError(f"Could not find a dataset containing {required_names}")

CLS_CKPT_DIR = find_ckpt_dir(["ConvNeXtV2Tiny_best.pt"])
SEG_CKPT_DIR = find_ckpt_dir(["DeepLabV3Plus_efficientnet-b3.pt", "UNet_resnet34.pt"])
heartbeat(f"CLS_CKPT_DIR = {CLS_CKPT_DIR}")
heartbeat(f"SEG_CKPT_DIR = {SEG_CKPT_DIR}")

CLS_CKPT = CLS_CKPT_DIR/"ConvNeXtV2Tiny_best.pt"
SEG_CKPTS = {
    "UNet_resnet34":                 SEG_CKPT_DIR/"UNet_resnet34.pt",
    "UNetPP_resnet50":               SEG_CKPT_DIR/"UNetPP_resnet50.pt",
    "FPN_resnet50":                  SEG_CKPT_DIR/"FPN_resnet50.pt",
    "DeepLabV3Plus_resnet50":        SEG_CKPT_DIR/"DeepLabV3Plus_resnet50.pt",
    "DeepLabV3Plus_efficientnet-b3": SEG_CKPT_DIR/"DeepLabV3Plus_efficientnet-b3.pt",
    "SegFormer_b2":                  SEG_CKPT_DIR/"SegFormer_b2.pt",
}

OUTDIR = Path("/kaggle/working/r2_notebookJ_results"); OUTDIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE_CLS = 384
IMG_SIZE_SEG = 512
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

meta = pd.read_csv(DATA_ROOT/"Metadata.csv", encoding="utf-8-sig")
meta.columns = [c.strip() for c in meta.columns]
meta["split"] = meta["Split"].map({"Training":"train","Validation":"val","Test":"test"})
meta["filepath"] = meta.apply(lambda r: DATA_ROOT/"images"/r["split"]/r["Name"], axis=1)
CLASS_NAMES = sorted(meta["Disease"].unique().tolist())
NUM_CLASSES = len(CLASS_NAMES)
test_df = meta[meta.split=="test"].reset_index(drop=True)
heartbeat(f"classes={NUM_CLASSES}  test={len(test_df)}")

from collections import defaultdict as _dd
from PIL import ImageDraw as _ID
MASK = Path("/kaggle/working/masks")
def load_anns(split):
    d = json.load(open(DATA_ROOT/f"annotation_{split}.json")); by = _dd(list)
    for a in d["annotations"]: by[a["image_id"]].append(a)
    fn = {}
    for im in d["images"]:
        polys = [seg for a in by.get(im["id"], []) for seg in a["segmentation"] if len(seg) >= 6]
        fn[im["file_name"]] = (im["width"], im["height"], polys)
    return fn
def rasterize(w, h, polys):
    m = Image.new("L", (w, h), 0); dr = _ID.Draw(m)
    for seg in polys: dr.polygon(list(zip(seg[0::2], seg[1::2])), fill=1)
    return m
fn = load_anns("test"); od = MASK/"test"; od.mkdir(parents=True, exist_ok=True)
mp_by_name = {}
for r in test_df.itertuples():
    w, h, polys = fn.get(r.Name, (None, None, []))
    mp = od/(Path(r.Name).stem + ".png")
    if not mp.exists():
        if w is None:
            with Image.open(r.filepath) as i0: w, h = i0.size
        rasterize(w, h, polys).save(mp)
    mp_by_name[r.Name] = mp
test_df["maskpath"] = test_df["Name"].map(mp_by_name)
heartbeat("test-split masks ready")

def load_state(model, ckpt_path):
    raw = torch.load(ckpt_path, map_location="cpu")
    if isinstance(raw, dict):
        for key in ("state_dict", "model_state_dict", "model", "best_state"):
            if key in raw and isinstance(raw[key], dict):
                raw = raw[key]; break
    raw = {k.replace("module.", ""): v for k, v in raw.items()}
    missing, unexpected = model.load_state_dict(raw, strict=False)
    if missing or unexpected:
        heartbeat(f"  [load_state:{Path(ckpt_path).name}] missing={len(missing)} unexpected={len(unexpected)}")
        total = len(model.state_dict())
        if total and len(missing) >= total * 0.5:
            raise RuntimeError(f"{Path(ckpt_path).name}: >=50% of keys unmatched -- "
                                f"checkpoint did not load; refusing to report numbers from random weights")
    return model

def build_classifier(num_classes):
    return timm.create_model("convnextv2_tiny", pretrained=False, num_classes=num_classes)

SMP_SPECS = {
    "UNet_resnet34":                 (smp.Unet,          "resnet34"),
    "UNetPP_resnet50":               (smp.UnetPlusPlus,  "resnet50"),
    "DeepLabV3Plus_resnet50":        (smp.DeepLabV3Plus, "resnet50"),
    "DeepLabV3Plus_efficientnet-b3": (smp.DeepLabV3Plus, "efficientnet-b3"),
    "FPN_resnet50":                  (smp.FPN,           "resnet50"),
}
SEGFORMER_HF = {"SegFormer_b2": "nvidia/mit-b2"}

class SegNet(nn.Module):
    def __init__(self, name, nc):
        super().__init__()
        if name in SMP_SPECS:
            arch, enc = SMP_SPECS[name]
            self.net = arch(encoder_name=enc, encoder_weights=None, in_channels=3, classes=nc); self.kind = "smp"
        else:
            self.net = SegformerForSemanticSegmentation.from_pretrained(
                SEGFORMER_HF[name], num_labels=nc, ignore_mismatched_sizes=True); self.kind = "hf"
    def forward(self, x):
        if self.kind == "smp": return self.net(x)
        lg = self.net(pixel_values=x).logits
        return F.interpolate(lg, size=x.shape[-2:], mode="bilinear", align_corners=False)

cls_tf = T.Compose([T.Resize((IMG_SIZE_CLS, IMG_SIZE_CLS)), T.ToTensor(), T.Normalize(MEAN, STD)])
seg_eval_tf = T.Compose([T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG)), T.ToTensor(), T.Normalize(MEAN, STD)])
mask_tf = T.Resize((IMG_SIZE_SEG, IMG_SIZE_SEG), interpolation=T.InterpolationMode.NEAREST)
heartbeat("config ready")

[15:51:20] DATA_ROOT = /kaggle/input/datasets/alenmaruf/plantseg/plantseg
[15:52:28] CLS_CKPT_DIR = /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/CLassification/43seed_classifier_checkpoints
[15:52:28] SEG_CKPT_DIR = /kaggle/input/datasets/alenmaruf/model-pt-class-seg-zip/model pt class+Seg/Segmentation/43seed_Seegmentation_checkpoints
[15:52:28] classes=115  test=1561
[15:52:45] test-split masks ready
[15:52:45] config ready


## The same 300-image stratified subset used throughout Tables 10-12

In [3]:
NAMES_300 = ['apple_black_rot_google_0037.jpg', 'apple_black_rot_google_0077.jpg', 'apple_mosaic_virus_57.jpg', 'apple_mosaic_virus_google_0074.jpg', 'apple_rust_147.jpg', 'apple_rust_google_0007.jpg', 'apple_rust_google_0011.jpg', 'apple_scab_50.jpg', 'apple_scab_85.jpg', 'apple_scab_google_0018.jpg', 'apple_scab_google_0031.jpg', 'apple_scab_google_0153.jpg', 'apple_scab_google_0244.jpg', 'banana_anthracnose_Baidu_0059.jpg', 'banana_anthracnose_Bing_0006.jpg', 'banana_anthracnose_Bing_0022.jpg', 'banana_anthracnose_Bing_0032.jpg', 'banana_black_leaf_streak_Bing_0124.jpg', 'banana_black_leaf_streak_banana black sigatoka (26).jpg', 'banana_bunchy_top_Bing_0152.jpg', 'banana_bunchy_top_Google_0037.jpg', 'banana_cigar_end_rot_Bing_0007.jpg', 'banana_cigar_end_rot_Google_0101.jpg', 'banana_cordana_leaf_spot_Bing_0007.jpg', 'banana_cordana_leaf_spot_Bing_0037.jpg', 'banana_panama_disease_126.jpg', 'banana_panama_disease_google_0064.jpg', 'basil_downy_mildew_google_0031.jpg', 'basil_downy_mildew_google_0091.jpg', 'bean_halo_blight_11.jpg', 'bean_halo_blight_2.jpg', 'bean_mosaic_virus_24.jpg', 'bean_mosaic_virus_google_0098.jpg', 'bean_rust_google_0045.jpg', 'bean_rust_google_0068.jpg', 'bell_pepper_bacterial_spot_Bing_0002.jpg', 'bell_pepper_bacterial_spot_Bing_0028.jpg', 'bell_pepper_blossom_end_rot_Bing_0002.jpg', 'bell_pepper_blossom_end_rot_Bing_0004.jpg', 'bell_pepper_blossom_end_rot_Bing_0077.jpg', 'bell_pepper_frogeye_leaf_spot_Bing_0008.jpg', 'bell_pepper_frogeye_leaf_spot_Google_0049.jpg', 'bell_pepper_powdery_mildew_Bing_0035.jpg', 'bell_pepper_powdery_mildew_Google_0118.jpg', 'blueberry_anthracnose_Bing_0006.jpg', 'blueberry_anthracnose_Bing_0145.jpg', 'blueberry_botrytis_blight_Bing_0015.jpg', 'blueberry_botrytis_blight_Bing_0021.jpg', 'blueberry_mummy_berry_Bing_0024.jpg', 'blueberry_mummy_berry_Google_0038.jpg', 'blueberry_rust_google_0004.jpg', 'blueberry_rust_google_0053.jpg', 'blueberry_scorch_Bing_0011.jpg', 'blueberry_scorch_Bing_0031.jpg', 'blueberry_scorch_Bing_0062.jpg', 'broccoli_alternaria_leaf_spot_Bing_0000.jpg', 'broccoli_alternaria_leaf_spot_Bing_0054.jpg', 'broccoli_downy_mildew_30.jpg', 'broccoli_downy_mildew_40.jpg', 'broccoli_ring_spot_Bing_0002.jpg', 'broccoli_ring_spot_Bing_0018.jpg', 'cabbage_alternaria_leaf_spot_3.jpg', 'cabbage_alternaria_leaf_spot_google_0206.jpg', 'cabbage_black_rot_Bing_0002.jpg', 'cabbage_black_rot_Bing_0018.jpg', 'cabbage_black_rot_Bing_0039.jpg', 'cabbage_black_rot_Bing_0080.jpg', 'cabbage_black_rot_Bing_0114.jpg', 'cabbage_downy_mildew_Bing_0012.jpg', 'cabbage_downy_mildew_Bing_0124.jpg', 'carrot_alternaria_leaf_blight_Bing_0046.jpg', 'carrot_alternaria_leaf_blight_Bing_0071.jpg', 'carrot_cavity_spot_google_0000.jpg', 'carrot_cavity_spot_google_0161.jpg', 'carrot_cercospora_leaf_blight_Bing_0009.jpg', 'carrot_cercospora_leaf_blight_Bing_0212.jpg', 'cauliflower_alternaria_leaf_spot_google_0003.jpg', 'cauliflower_alternaria_leaf_spot_google_0061.jpg', 'cauliflower_bacterial_soft_rot_Bing_0021.jpg', 'cauliflower_bacterial_soft_rot_Bing_0205.jpg', 'celery_anthracnose_google_0007.jpg', 'celery_anthracnose_google_0035.jpg', 'celery_early_blight_52.jpg', 'celery_early_blight_google_0001.jpg', 'celery_early_blight_google_0009.jpg', 'cherry_leaf_spot_google_0180.jpg', 'cherry_leaf_spot_google_0216.jpg', 'cherry_powdery_mildew_107.jpg', 'cherry_powdery_mildew_2.jpg', 'citrus_canker_13.jpg', 'citrus_canker_147.jpg', 'citrus_canker_163.jpg', 'citrus_canker_247.jpg', 'citrus_canker_363.jpg', 'citrus_canker_76.jpg', 'citrus_canker_89.jpg', 'citrus_canker_google_0071.jpg', 'citrus_greening_disease_29.jpg', 'citrus_greening_disease_48.jpg', 'citrus_greening_disease_google_0120.jpg', 'citrus_greening_disease_google_0251.jpg', 'coffee_berry_blotch_Bing_0027.jpg', 'coffee_berry_blotch_Bing_0114.jpg', 'coffee_berry_blotch_Bing_0120.jpg', 'coffee_brown_eye_spot_Bing_0036.jpg', 'coffee_brown_eye_spot_Google_0003.jpg', 'coffee_leaf_rust_15.jpg', 'coffee_leaf_rust_google_0016.jpg', 'coffee_leaf_rust_google_0135.jpg', 'corn_gray_leaf_spot_31.jpg', 'corn_gray_leaf_spot_35.jpg', 'corn_gray_leaf_spot_google_0074.jpg', 'corn_northern_leaf_blight_101.jpg', 'corn_northern_leaf_blight_14.jpg', 'corn_northern_leaf_blight_81.jpg', 'corn_rust_80.jpg', 'corn_rust_google_0078.jpg', 'corn_smut_google_0013.jpg', 'corn_smut_google_0063.jpg', 'corn_smut_google_0079.jpg', 'cucumber_angular_leaf_spot_1.jpg', 'cucumber_angular_leaf_spot_156.jpg', 'cucumber_angular_leaf_spot_328.jpg', 'cucumber_angular_leaf_spot_41.jpg', 'cucumber_angular_leaf_spot_google_0043.jpg', 'cucumber_bacterial_wilt_109.jpg', 'cucumber_bacterial_wilt_313.jpg', 'cucumber_bacterial_wilt_50.jpg', 'cucumber_powdery_mildew_58.jpg', 'cucumber_powdery_mildew_70.jpg', 'cucumber_powdery_mildew_77.jpg', 'cucumber_powdery_mildew_google_0159.jpg', 'eggplant_cercospora_leaf_spot_12.jpg', 'eggplant_cercospora_leaf_spot_33.jpg', 'eggplant_cercospora_leaf_spot_40.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0016.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0022.jpg', 'eggplant_phomopsis_fruit_rot_Bing_0261.jpg', 'eggplant_phytophthora_blight_Bing_0000.jpg', 'eggplant_phytophthora_blight_Bing_0002.jpg', 'garlic_leaf_blight_140.jpg', 'garlic_leaf_blight_46.jpg', 'garlic_leaf_blight_google_0004.jpg', 'garlic_leaf_blight_google_0210.jpg', 'garlic_rust_159.jpg', 'garlic_rust_18.jpg', 'ginger_leaf_spot_11.jpg', 'ginger_leaf_spot_google_0001.jpg', 'ginger_sheath_blight_53.jpg', 'ginger_sheath_blight_89.jpg', 'grape_black_rot_12.jpg', 'grape_black_rot_216.jpg', 'grape_black_rot_google_0013.jpg', 'grape_downy_mildew_251.jpg', 'grape_downy_mildew_272.jpg', 'grape_downy_mildew_google_0032.jpg', 'grape_downy_mildew_google_0052.jpg', 'grape_leaf_spot_101.jpg', 'grape_leaf_spot_168.jpg', 'grapevine_leafroll_disease_google_0001.jpg', 'grapevine_leafroll_disease_google_0181.jpg', 'lettuce_downy_mildew_google_0015.jpg', 'lettuce_downy_mildew_google_0027.jpg', 'lettuce_mosaic_virus_16.jpg', 'lettuce_mosaic_virus_24.jpg', 'maple_tar_spot_google_0005.jpg', 'maple_tar_spot_google_0089.jpg', 'peach_anthracnose_Bing_0002.jpg', 'peach_anthracnose_Bing_0013.jpg', 'peach_brown_rot_Bing_0000.jpg', 'peach_brown_rot_Bing_0092.jpg', 'peach_brown_rot_Bing_0114.jpg', 'peach_leaf_curl_34.jpg', 'peach_leaf_curl_google_0020.jpg', 'peach_leaf_curl_google_0035.jpg', 'peach_leaf_curl_google_0074.jpg', 'peach_rust_Bing_0001.jpg', 'peach_scab_Bing_0007.jpg', 'peach_scab_Bing_0083.jpg', 'plum_bacterial_spot_Bing_0013.jpg', 'plum_bacterial_spot_Bing_0082.jpg', 'plum_brown_rot_Bing_0001.jpg', 'plum_brown_rot_Bing_0097.jpg', 'plum_pocket_disease_google_0028.jpg', 'plum_pocket_disease_google_0032.jpg', 'plum_pocket_disease_google_0117.jpg', 'plum_pox_virus_Bing_0052.jpg', 'plum_pox_virus_Bing_0054.jpg', 'plum_pox_virus_Google_0040.jpg', 'plum_rust_Bing_0024.jpg', 'plum_rust_Google_0036.jpg', 'potato_early_blight_6.jpg', 'potato_early_blight_google_0095.jpg', 'potato_late_blight_11.jpg', 'potato_late_blight_36.jpg', 'potato_late_blight_96.jpg', 'raspberry_fire_blight_Bing_0018.jpg', 'raspberry_fire_blight_Google_0005.jpg', 'raspberry_gray_mold_Bing_0000.jpg', 'raspberry_gray_mold_Bing_0001.jpg', 'raspberry_leaf_spot_Bing_0073.jpg', 'raspberry_yellow_rust_Bing_0010.jpg', 'raspberry_yellow_rust_Bing_0183.jpg', 'rice_blast_151.jpg', 'rice_blast_87.jpg', 'rice_sheath_blight_148.jpg', 'rice_sheath_blight_google_0027.jpg', 'soybean_bacterial_blight_Bing_0057.jpg', 'soybean_bacterial_blight_Bing_0105.jpg', 'soybean_bacterial_blight_Bing_0172.jpg', 'soybean_brown_spot_Baidu_0010.jpg', 'soybean_brown_spot_Bing_0353.jpg', 'soybean_downy_mildew_Bing_0078.jpg', 'soybean_downy_mildew_Bing_0143.jpg', 'soybean_frog_eye_leaf_spot_Bing_0141.jpg', 'soybean_frog_eye_leaf_spot_Google_0057.jpg', 'soybean_frog_eye_leaf_spot_Google_0221.jpg', 'soybean_mosaic_Baidu_0059.jpg', 'soybean_mosaic_Baidu_0089.jpg', 'soybean_rust_Bing_0022.jpg', 'soybean_rust_Bing_0055.jpg', 'soybean_rust_Bing_0074.jpg', 'soybean_rust_Bing_0075.jpg', 'soybean_rust_Bing_0096.jpg', 'squash_powdery_mildew_129.jpg', 'squash_powdery_mildew_23.jpg', 'squash_powdery_mildew_6.jpg', 'squash_powdery_mildew_google_0105.jpg', 'strawberry_anthracnose_5.jpg', 'strawberry_anthracnose_google_0001.jpg', 'strawberry_leaf_scorch_google_0037.jpg', 'strawberry_leaf_scorch_google_0133.jpg', 'tobacco_blue_mold_Bing_0024.jpg', 'tobacco_blue_mold_Bing_0096.jpg', 'tobacco_brown_spot_Bing_0001.jpg', 'tobacco_brown_spot_Bing_0047.jpg', 'tobacco_brown_spot_Bing_0055.jpg', 'tobacco_frogeye_leaf_spot_Bing_0002.jpg', 'tobacco_frogeye_leaf_spot_Google_0052.jpg', 'tobacco_mosaic_virus_117.jpg', 'tobacco_mosaic_virus_4.jpg', 'tomato_bacterial_leaf_spot_120.jpg', 'tomato_bacterial_leaf_spot_14.jpg', 'tomato_early_blight_96.jpg', 'tomato_early_blight_google_0059.jpg', 'tomato_early_blight_google_0177.jpg', 'tomato_late_blight_10.jpg', 'tomato_late_blight_182.jpg', 'tomato_late_blight_205.jpg', 'tomato_leaf_mold_10.jpg', 'tomato_leaf_mold_148.jpg', 'tomato_mosaic_virus_15.jpg', 'tomato_mosaic_virus_google_0049.jpg', 'tomato_septoria_leaf_spot_10.jpg', 'tomato_septoria_leaf_spot_18.jpg', 'tomato_septoria_leaf_spot_243.jpg', 'tomato_septoria_leaf_spot_80.jpg', 'tomato_septoria_leaf_spot_google_0027.jpg', 'tomato_septoria_leaf_spot_google_0070.jpg', 'tomato_septoria_leaf_spot_google_0128.jpg', 'tomato_yellow_leaf_curl_virus_15.jpg', 'tomato_yellow_leaf_curl_virus_17.jpg', 'tomato_yellow_leaf_curl_virus_40.jpg', 'tomato_yellow_leaf_curl_virus_99.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0031.jpg', 'wheat_bacterial_leaf_streak_(black_chaff)_Baidu_0235.jpg', 'wheat_head_scab_Baidu_0481.jpg', 'wheat_head_scab_Bing_0040.jpg', 'wheat_head_scab_Bing_0043.jpg', 'wheat_head_scab_Bing_0288.jpg', 'wheat_leaf_rust_Google_0040.jpg', 'wheat_leaf_rust_Google_0058.jpg', 'wheat_leaf_rust_Google_0328.jpg', 'wheat_loose_smut_Bing_0109.jpg', 'wheat_loose_smut_Bing_0283.jpg', 'wheat_loose_smut_Google_0081.jpg', 'wheat_powdery_mildew_Baidu_0181.jpg', 'wheat_powdery_mildew_Bing_0014.jpg', 'wheat_powdery_mildew_Google_0014.jpg', 'wheat_septoria_blotch_Google_0115.jpg', 'wheat_septoria_blotch_google_septoria (18).jpg', 'wheat_stem_rust_Baidu_0001.jpg', 'wheat_stem_rust_Bing_0217.jpg', 'wheat_stem_rust_Google_0274.jpg', 'wheat_stripe_rust_Baidu_0143.jpg', 'wheat_stripe_rust_Bing_0007.jpg', 'wheat_stripe_rust_Bing_0257.jpg', 'wheat_stripe_rust_Google_0029.jpg', 'zucchini_bacterial_wilt_Bing_0105.jpg', 'zucchini_bacterial_wilt_Bing_0188.jpg', 'zucchini_bacterial_wilt_Bing_0260.jpg', 'zucchini_bacterial_wilt_Google_0060.jpg', 'zucchini_downy_mildew_Bing_0035.jpg', 'zucchini_downy_mildew_Bing_0259.jpg', 'zucchini_powdery_mildew_Bing_0109.jpg', 'zucchini_powdery_mildew_Bing_0155.jpg', 'zucchini_powdery_mildew_Bing_0233.jpg', 'zucchini_yellow_mosaic_virus_11.jpg', 'zucchini_yellow_mosaic_virus_google_0038.jpg', 'zucchini_yellow_mosaic_virus_google_0067.jpg']

sub_df = test_df[test_df["Name"].isin(NAMES_300)].reset_index(drop=True)
heartbeat(f"matched {len(sub_df)} / {len(NAMES_300)} images of the subset actually present "
          f"in this checkpoint of Metadata.csv -- if below 300, this truncated NAMES_300 list "
          f"(copy-length-limited when this notebook was built) still gives a valid, if smaller, "
          f"paired comparison; the Wilcoxon test and all ratios below are unaffected by subset size.")
assert len(sub_df) > 0, 'no images matched -- check the Name column / dataset version'

[15:52:45] matched 300 / 300 images of the subset actually present in this checkpoint of Metadata.csv -- if below 300, this truncated NAMES_300 list (copy-length-limited when this notebook was built) still gives a valid, if smaller, paired comparison; the Wilcoxon test and all ratios below are unaffected by subset size.


## Shared metric functions (identical to NotebookA's chance-baseline check) + the new centre-prior generator

In [4]:
def cam_gt_metrics(cam, gt_mask):
    if gt_mask.sum() == 0: return {"iou": np.nan, "pointing_hit": np.nan}
    thresh = np.quantile(cam, 1 - gt_mask.mean())
    cam_bin = cam >= thresh
    inter = (cam_bin & gt_mask).sum(); union = (cam_bin | gt_mask).sum()
    iou = float(inter/union) if union > 0 else np.nan
    py, px = np.unravel_index(np.argmax(cam), cam.shape)
    return {"iou": iou, "pointing_hit": bool(gt_mask[py, px])}

CENTRE_SIGMA_FRAC = 0.2  # stated design choice: sigma = 0.2 * min(H, W); not fit to data

def centre_prior_map(h, w, sigma_frac=CENTRE_SIGMA_FRAC):
    """Fixed 2D Gaussian centred on the frame -- the 'always look at the middle of the
    photo' heuristic. Deterministic given (h, w): does not depend on the model or the
    image content, only its size, matching the paper's own description that this
    baseline is 'architecture-independent, so identical for every row.'"""
    yy, xx = np.mgrid[0:h, 0:w]
    cy, cx = (h - 1) / 2.0, (w - 1) / 2.0
    sigma = sigma_frac * min(h, w)
    g = np.exp(-(((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * sigma ** 2)))
    return g / (g.max() + 1e-8)

def rng_random_map(h, w, rng):
    return rng.rand(h, w)

RNG = np.random.RandomState(SEED)
heartbeat("metric functions ready (cam_gt_metrics verbatim from NotebookA; centre_prior_map is new)")

[15:52:45] metric functions ready (cam_gt_metrics verbatim from NotebookA; centre_prior_map is new)


## Section 1 -- Classifier (ConvNeXtV2-Tiny): real Grad-CAM vs. random vs. centre-prior,
## plus the model-parameter-randomisation sanity check

In [5]:
clf = build_classifier(NUM_CLASSES); clf = load_state(clf, CLS_CKPT).to(device).eval()
cam_extractor = GradCAM(model=clf, target_layers=[clf.stages[-1]])

clf_rand_init = build_classifier(NUM_CLASSES).to(device).eval()  # fresh random weights, no checkpoint loaded
cam_extractor_rand = GradCAM(model=clf_rand_init, target_layers=[clf_rand_init.stages[-1]])

clf_rows = []
t0 = time.time()
for i, r in enumerate(sub_df.itertuples()):
    img = Image.open(r.filepath).convert("RGB")
    gt = np.array(Image.open(r.maskpath)); gt_bin = (gt > 0)
    x = cls_tf(img).unsqueeze(0).to(device)

    cam = cam_extractor(input_tensor=x, targets=None)[0]
    cam_rs = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(gt_bin.shape[::-1])) / 255.0
    real_m = cam_gt_metrics(cam_rs, gt_bin)

    cam_r = cam_extractor_rand(input_tensor=x, targets=None)[0]
    cam_r_rs = np.array(Image.fromarray((cam_r*255).astype(np.uint8)).resize(gt_bin.shape[::-1])) / 255.0
    rand_init_m = cam_gt_metrics(cam_r_rs, gt_bin)

    rmap = rng_random_map(*gt_bin.shape, RNG)
    random_m = cam_gt_metrics(rmap, gt_bin)

    cmap = centre_prior_map(*gt_bin.shape)
    centre_m = cam_gt_metrics(cmap, gt_bin)

    clf_rows.append({"model": "ConvNeXtV2Tiny_classifier", "image": r.Name,
                      "real_iou": real_m["iou"], "real_hit": real_m["pointing_hit"],
                      "randinit_iou": rand_init_m["iou"], "randinit_hit": rand_init_m["pointing_hit"],
                      "random_iou": random_m["iou"], "random_hit": random_m["pointing_hit"],
                      "centre_iou": centre_m["iou"], "centre_hit": centre_m["pointing_hit"]})
    if (i+1) % 50 == 0:
        heartbeat(f"classifier: {i+1}/{len(sub_df)} images")

del clf, clf_rand_init, cam_extractor, cam_extractor_rand; torch.cuda.empty_cache()
clf_df = pd.DataFrame(clf_rows)
heartbeat(f"classifier section done in {time.time()-t0:.1f}s")

[15:53:04] classifier: 50/300 images
[15:53:15] classifier: 100/300 images
[15:53:25] classifier: 150/300 images
[15:53:36] classifier: 200/300 images
[15:53:47] classifier: 250/300 images
[15:53:58] classifier: 300/300 images
[15:53:58] classifier section done in 69.9s


## Section 2 -- Five CNN segmenters + SegFormer-B2 (pinned transformers): real vs. random vs. centre-prior

In [6]:
class _GradCAMHook:
    def __init__(self, module):
        self.activations = None; self.gradients = None
        self.fh = module.register_forward_hook(self._fwd)
        self.bh = module.register_full_backward_hook(self._bwd)
    def _fwd(self, m, inp, out): self.activations = out
    def _bwd(self, m, gi, go):  self.gradients  = go[0]
    def remove(self): self.fh.remove(); self.bh.remove()

def gradcam_map(model, x, nc):
    hook = _GradCAMHook(model.net.decoder)
    model.zero_grad(set_to_none=True)
    logits = model(x)
    score = logits[:,1] if nc==2 else logits[:,1:].sum(1)
    score.sum().backward()
    acts  = hook.activations[0].detach(); grads = hook.gradients[0].detach(); hook.remove()
    w = grads.mean(dim=(1,2))
    cam = torch.relu((w[:,None,None]*acts).sum(0)); cam = cam/(cam.max()+1e-8)
    cam = F.interpolate(cam[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def segformer_attention_map(model, x):
    with torch.no_grad():
        out = model.net.segformer(pixel_values=x, output_attentions=True) \
              if hasattr(model.net, "segformer") else model.net(pixel_values=x, output_attentions=True)
    attns = out.attentions
    if not attns:
        raise RuntimeError("segformer_attention_map: no attentions returned -- likely a failed "
                            "checkpoint load or transformers version mismatch")
    stages=[]; cur=[]; cur_n=None
    for a in attns:
        n=a.shape[-2]
        if cur_n is None or n==cur_n: cur.append(a); cur_n=n
        else: stages.append(cur); cur=[a]; cur_n=n
    stages.append(cur)
    rollout=None
    for a in stages[-1]:
        A = a[0].mean(0); A = A/A.sum(-1,keepdim=True).clamp_min(1e-8)
        A_hat = 0.5*A + 0.5*torch.eye(A.shape[0], device=A.device)
        rollout = A_hat if rollout is None else A_hat@rollout
    imp = rollout.mean(0); side = int(round(imp.shape[0]**0.5))
    grid = imp[:side*side].reshape(side,side).float(); grid = grid/(grid.max()+1e-8)
    cam  = F.interpolate(grid[None,None], size=x.shape[-2:], mode="bilinear", align_corners=False)[0,0]
    return cam.detach().cpu().numpy()

def explain(model, x, nc, is_transformer):
    return segformer_attention_map(model,x) if is_transformer else gradcam_map(model,x,nc)

seg_rows = []
for name, ckpt in SEG_CKPTS.items():
    t0 = time.time()
    try:
        model = SegNet(name, 2); model = load_state(model, ckpt).to(device).eval()
    except Exception as e:
        heartbeat(f"  [SKIP] {name}: checkpoint failed to load -- {e}"); continue
    is_tf = (name == "SegFormer_b2")
    n_ok = 0
    for i, r in enumerate(sub_df.itertuples()):
        try:
            x = seg_eval_tf(Image.open(r.filepath).convert("RGB")).unsqueeze(0).to(device)
            gt = np.array(mask_tf(Image.open(r.maskpath))) > 0
            cam = explain(model, x, 2, is_tf)
            real_m = cam_gt_metrics(cam, gt)
            rmap = rng_random_map(*gt.shape, RNG)
            random_m = cam_gt_metrics(rmap, gt)
            cmap = centre_prior_map(*gt.shape)
            centre_m = cam_gt_metrics(cmap, gt)
            seg_rows.append({"model": name, "image": r.Name,
                              "real_iou": real_m["iou"], "real_hit": real_m["pointing_hit"],
                              "random_iou": random_m["iou"], "random_hit": random_m["pointing_hit"],
                              "centre_iou": centre_m["iou"], "centre_hit": centre_m["pointing_hit"]})
            n_ok += 1
        except Exception as e:
            heartbeat(f"  [SKIP image] {name} / {r.Name}: {e}"); continue
        if (i+1) % 100 == 0:
            heartbeat(f"  {name}: {i+1}/{len(sub_df)} images")
    del model; torch.cuda.empty_cache()
    if n_ok == 0:
        heartbeat(f"  [SKIP] {name}: 0/{len(sub_df)} images succeeded -- excluded from results")
    else:
        heartbeat(f"  done: {name} ({n_ok}/{len(sub_df)} images) in {time.time()-t0:.1f}s")

seg_df = pd.DataFrame(seg_rows)
heartbeat("segmenter section done (SegFormer-B2 included if the transformers pin worked above)")

[15:54:09]   UNet_resnet34: 100/300 images
[15:54:18]   UNet_resnet34: 200/300 images
[15:54:27]   UNet_resnet34: 300/300 images
[15:54:27]   done: UNet_resnet34 (300/300 images) in 28.7s
[15:55:03]   UNetPP_resnet50: 100/300 images
[15:55:35]   UNetPP_resnet50: 200/300 images
[15:56:07]   UNetPP_resnet50: 300/300 images
[15:56:07]   done: UNetPP_resnet50 (300/300 images) in 99.9s
[15:56:19]   FPN_resnet50: 100/300 images
[15:56:28]   FPN_resnet50: 200/300 images
[15:56:38]   FPN_resnet50: 300/300 images
[15:56:38]   done: FPN_resnet50 (300/300 images) in 31.2s
[15:56:52]   DeepLabV3Plus_resnet50: 100/300 images
[15:57:04]   DeepLabV3Plus_resnet50: 200/300 images
[15:57:16]   DeepLabV3Plus_resnet50: 300/300 images
[15:57:16]   done: DeepLabV3Plus_resnet50 (300/300 images) in 37.5s
[15:57:28]   DeepLabV3Plus_efficientnet-b3: 100/300 images
[15:57:39]   DeepLabV3Plus_efficientnet-b3: 200/300 images
[15:57:49]   DeepLabV3Plus_efficientnet-b3: 300/300 images
[15:57:49]   done: DeepLabV3Plu

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/99.0M [00:00<?, ?B/s]

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[15:58:00]   SegFormer_b2: 100/300 images
[15:58:06]   SegFormer_b2: 200/300 images
[15:58:12]   SegFormer_b2: 300/300 images
[15:58:12]   done: SegFormer_b2 (300/300 images) in 22.5s
[15:58:12] segmenter section done (SegFormer-B2 included if the transformers pin worked above)


## Aggregate: Table-12-shaped summary (real / random / centre-prior, paired Wilcoxon), plus the randomisation check

In [7]:
def summarize(df, real_col, other_col, hit_real_col, hit_other_col, label):
    a = df[real_col].dropna()
    paired = df.loc[a.index]
    b = paired[other_col]
    hit_a = paired[hit_real_col]; hit_b = paired[hit_other_col]
    try:
        stat_i, p_i = wilcoxon(a, b)
    except Exception:
        p_i = np.nan
    try:
        stat_h, p_h = wilcoxon(hit_a.astype(float), hit_b.astype(float))
    except Exception:
        p_h = np.nan
    return {"comparison": label, "n": len(a),
            "real_mean_iou": float(a.mean()), "other_mean_iou": float(b.mean()), "wilcoxon_p_iou": float(p_i),
            "real_mean_hit": float(hit_a.mean()), "other_mean_hit": float(hit_b.mean()), "wilcoxon_p_hit": float(p_h)}

table12_rows = []
for name, grp in pd.concat([clf_df.assign(model="ConvNeXtV2Tiny_classifier"), seg_df]).groupby("model"):
    row_random = summarize(grp, "real_iou", "random_iou", "real_hit", "random_hit", f"{name} vs random")
    row_centre = summarize(grp, "real_iou", "centre_iou", "real_hit", "centre_hit", f"{name} vs centre")
    table12_rows.append({
        "model": name, "n": row_random["n"],
        "real_iou": row_random["real_mean_iou"], "p_vs_random_iou": row_random["wilcoxon_p_iou"],
        "p_vs_centre_iou": row_centre["wilcoxon_p_iou"],
        "real_hit": row_random["real_mean_hit"], "p_vs_random_hit": row_random["wilcoxon_p_hit"],
        "p_vs_centre_hit": row_centre["wilcoxon_p_hit"],
        "random_mean_iou": row_random["other_mean_iou"], "random_mean_hit": row_random["other_mean_hit"],
        "centre_mean_iou": row_centre["other_mean_iou"], "centre_mean_hit": row_centre["other_mean_hit"],
    })
table12_summary = pd.DataFrame(table12_rows)
print("=" * 100)
print("TABLE 12 (tab:nullmodels) -- recomputed for real")
print("=" * 100)
print(table12_summary.round(4).to_string(index=False))

randomisation_summary = summarize(clf_df, "real_iou", "randinit_iou", "real_hit", "randinit_hit",
                                   "ConvNeXtV2Tiny: trained vs. fully-randomly-initialised weights")
print()
print("=" * 100)
print("MODEL-PARAMETER RANDOMISATION SANITY CHECK (response letter, R4 Concern 2)")
print("=" * 100)
for k, v in randomisation_summary.items():
    print(f"  {k}: {v}")

heartbeat("all comparisons computed")

TABLE 12 (tab:nullmodels) -- recomputed for real
                        model   n  real_iou  p_vs_random_iou  p_vs_centre_iou  real_hit  p_vs_random_hit  p_vs_centre_hit  random_mean_iou  random_mean_hit  centre_mean_iou  centre_mean_hit
    ConvNeXtV2Tiny_classifier 300    0.2629              0.0           0.2411    0.5133              0.0           0.2123           0.1358           0.1900           0.2804           0.4667
DeepLabV3Plus_efficientnet-b3 300    0.4694              0.0           0.0000    0.8867              0.0           0.0000           0.1359           0.1933           0.2839           0.4667
       DeepLabV3Plus_resnet50 300    0.4908              0.0           0.0000    0.8800              0.0           0.0000           0.1358           0.1767           0.2839           0.4667
                 FPN_resnet50 300    0.3946              0.0           0.0000    0.8933              0.0           0.0000           0.1358           0.1867           0.2839           0.4667
 

## Save everything

In [8]:
clf_df.to_csv(OUTDIR/"notebookJ_classifier_per_image.csv", index=False)
seg_df.to_csv(OUTDIR/"notebookJ_segmenters_per_image.csv", index=False)
table12_summary.to_csv(OUTDIR/"notebookJ_table12_SUMMARY.csv", index=False)
with open(OUTDIR/"notebookJ_randomisation_check.json", "w") as f:
    json.dump(randomisation_summary, f, indent=2)

import pickle
with open(OUTDIR/"notebookJ_full_record.pkl", "wb") as f:
    pickle.dump({"table12_summary": table12_summary.to_dict(orient="records"),
                 "randomisation_summary": randomisation_summary,
                 "centre_sigma_frac": CENTRE_SIGMA_FRAC,
                 "transformers_version": _tf.__version__,
                 "n_subset_images": len(sub_df),
                 "run_timestamp": time.strftime("%Y-%m-%d %H:%M:%S")}, f)

import zipfile as _zf
zip_path = Path("/kaggle/working/plantseg_r2_notebookJ_results.zip")
with _zf.ZipFile(zip_path, "w", _zf.ZIP_DEFLATED) as zf:
    for p in sorted(OUTDIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
heartbeat(f"SAVED {zip_path.name} ({zip_path.stat().st_size/1024:.1f} KB)")
heartbeat("Notebook J complete. Bring the CSVs/JSON (or the zip) back for the paper/letter update.")

[15:58:12] SAVED plantseg_r2_notebookJ_results.zip (81.4 KB)
[15:58:12] Notebook J complete. Bring the CSVs/JSON (or the zip) back for the paper/letter update.
